# Preprocessing v3 — Pruning Anomaly Filter
Adding the Near_Pruning_Flag filter. Post-pruning recovery fields have Target_Days of 37-150 days — biologically distinct and never seen in training data.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25], ignore_index=True)
df_test  = df26.copy()

for df in [df_train,df_test]:
    df.drop(columns=[c for c in LEAK if c in df.columns], inplace=True)

print(f"Before pruning filter — Train: {df_train.shape[0]}  Test: {df_test.shape[0]}")

# Apply pruning filter
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_train.drop(columns=["Near_Pruning_Flag"], inplace=True)
df_test.drop(columns=["Near_Pruning_Flag"],  inplace=True)

print(f"After  pruning filter — Train: {df_train.shape[0]}  Test: {df_test.shape[0]}")
print(f"Removed from test: 9 post-pruning recovery fields")


In [ ]:
TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(df_train[num_cols])
X_test  = imp.transform(df_test[num_cols])
y_train = df_train[TARGET].values
y_test  = df_test[TARGET].values

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"y_train range: {y_train.min():.2f} — {y_train.max():.2f} days")
print(f"y_test  range: {y_test.min():.2f} — {y_test.max():.2f} days")
print("\nImprovements so far: leakage removed, pruning filter applied")
print("Still missing: feature engineering, outlier clipping, proper scaling")


## Status: v3 — Pruning Filter Added
- Test set: 71 → 62 clean rows
- Training set stays at 141 (no flagged rows in 2024/2025)
- Removes biologically abnormal fields that model has never seen patterns for